# MorphIQ SDK for Python — American pricing

Price an American put with a continuous dividend yield. Package the option inputs, let the service select the model, and read its result and charge evidence. The example uses spot 100, strike 105, half a year to expiry, 20% volatility, a 5% interest rate and a 1% continuous dividend yield.

The inputs are synthetic. This notebook runs independently with the SDK and Jupyter installed; it does not import repository helpers.

Run All makes **1 API request**, which may consume your API allowance. Opening the notebook makes no API calls.

[Back to the quickstart](../start-here.ipynb)


In [ ]:
import sys

import morphiq_sdk

print("Python:", sys.version.split()[0])
print("MorphIQ SDK:", morphiq_sdk.__version__)

## Connect to your API

Get your API key from the
[MorphIQ Labs developer portal](https://developers.morphiqlabs.com).
The SDK connects to `https://api.morphiqlabs.com` automatically. Before starting
Jupyter or Docker Compose, set your API key in the terminal:

```sh
export MORPHIQ_API_KEY="YOUR_API_KEY"
```

Docker Compose passes the key into the notebook container. If it is unset,
the cell below prompts. The key prompt is masked: **do not paste your key into a code
cell**. The key stays in this kernel's process environment, not notebook source
or outputs; clear it in the cleanup cell and shut down the kernel when finished.

For a local or controlled test service, set `MORPHIQ_API_BASE_URL` before startup
to override the default API URL.

Each example cell contains a complete example: request inputs, API call, response
fields and error handling. The code is copied from the executable Python scripts.
The server validates input domains and performs pricing or regime inference.


In [ ]:
import os
from getpass import getpass

if not os.environ.get("MORPHIQ_API_KEY"):
    os.environ["MORPHIQ_API_KEY"] = getpass("MorphIQ API key: ")

In [ ]:
import os
import sys
from getpass import getpass

from morphiq_sdk import APIError, MorphIQ, PriceRequest, ResponseError, TransportError

# Set MORPHIQ_API_KEY in your environment, or enter it at the masked prompt.
api_key = os.environ.get("MORPHIQ_API_KEY") or getpass("MorphIQ API key: ")

# Synthetic American put; spot and strike use the same currency.
request = PriceRequest(
    exercise_style="american",
    option_type="put",
    spot=100.0,
    strike=105.0,
    time_to_expiry_years=0.5,  # Years from valuation.
    risk_free_rate=0.05,  # 5% annual rate.
    dividend_yield=0.01,
    volatility=0.20,  # 20% annual volatility.
    pricing_model="auto",
)

try:
    # The client uses https://api.morphiqlabs.com and closes its connection pool here.
    with MorphIQ(api_key=api_key) as client:
        response = client.price(request)
except APIError as error:
    print(f"HTTP {error.status_code}: {error.code}: {error.message}", file=sys.stderr)
    print(f"Request ID: {error.request_id}", file=sys.stderr)
    charged = error.metadata.fcu_charged if error.metadata else None
    print(
        f"Commercial CU charged: {charged if charged is not None else 'not reported'}",
        file=sys.stderr,
    )
    raise SystemExit(1) from None
except TransportError:
    # Includes timeouts. The service may already have committed a charge.
    raise SystemExit(
        "No complete response; execution and charges may be unknown. No retry."
    ) from None
except ResponseError as error:
    raise SystemExit(
        f"Invalid response; request_id={error.request_id}. Charges may be unknown. No retry."
    ) from None

print(f"Price: {response.result.price}")
print(f"Model: {response.result.pricing_model} ({response.result.model_version})")
print(f"Request ID: {response.metadata.request_id}")
for engine in response.metadata.engines:
    print(f"Engine: {engine.name} {engine.version} ({engine.revision})")
# Missing commercial CU is unreported, not zero.
charged = response.metadata.fcu_charged
print("Commercial CU charged:", charged if charged is not None else "not reported")

## Reading results and failures

Use `response.result.price`, `response.result.pricing_model`, and
`response.metadata.request_id`. Commercial charges are `metadata.fcu_charged`;
if absent they are **unreported**, not zero. `units_charged` is a separate
internal compute weight. Keep the request ID when contacting support.

`APIError` exposes HTTP status, server code, message, request ID, optional charge
metadata and `Retry-After`. `RequestTimeout` means execution/charges may be
unknown. Both clients make **no automatic retries**, including after timeouts,
and do not follow redirects. Correct the input/key/endpoint or inspect the
reported charge before deliberately submitting another request.

## Finish

The next cell removes the API key from this kernel's environment. Shut down the
kernel to release its remaining memory. If you exported the key in your terminal,
run `unset MORPHIQ_API_KEY` there when finished; `docker compose down` removes the
container configured with that key. Clear outputs before sharing a notebook.


In [ ]:
if "MORPHIQ_API_KEY" in os.environ:
    del os.environ["MORPHIQ_API_KEY"]
print("API key removed from the kernel environment.")